# Método CASTLE · Notebook 3: Ejecución
**Castle Quant** — Del modelo y la estrategia a un sistema que procesa los datos uno a uno, y de ahí a un bot conectado a Binance Demo.

In [1]:
# y_hat = model(x)
# orders = strategy(y_hat)
# execute(orders)

In [2]:
# Notebook 1 = C · Captura y A · Alfa                            (Investigación)
# Notebook 2 = S · Simulación, T · Tamaño y L · Libre de sesgos  (Estrategia)
# Notebook 3 = E · Ejecución                                     (Implementación)

## Objetivos

In [3]:
# 1. Programarlo todo junto en Python (Modelo + Estrategia)
# 2. Mostrar cómo construir un sistema de trading escalable (muy fácil crear estrategias nuevas + datos en streaming)
# 3. Conectarlo a una cuenta demo de Binance, con dinero ficticio, para verlo operar solo

In [4]:
# este notebook se ejecuta en tu computador (VS Code o Jupyter), no en Colab: el bot necesita quedarse encendido
# y tus claves no deben salir de tu equipo

## Repaso

### Notebook 1: Captura y Alfa

In [5]:
import models
import torch
import research

model = models.LinearModel(2)
model.load_state_dict(torch.load('model_weights.pth', weights_only=True))
model.eval()

LinearModel(
  (linear): Linear(in_features=2, out_features=1, bias=True)
)

### Modelo lineal para predecir el retorno logarítmico futuro - horizonte de 12h, rezagos 1 y 3

In [6]:
research.print_model_params(model)

linear.weight:
[[-0.00699938  0.01524278]]
linear.bias:
[0.0002775]


### Notebook 2: Simulación, Tamaño y Libre de sesgos

In [7]:
# la regla: cada 12 horas, todo el capital en la dirección que predice el modelo; solo se opera cuando la dirección cambia
# el veredicto: el modelo no tiene una ventaja demostrada. El sistema que construimos hoy no depende de eso:
# funciona con cualquier modelo. Funcional no es lo mismo que rentable

### Bloque Fundamental: Tick

In [8]:
from abc import ABC, abstractmethod
from typing import Generic, TypeVar

T = TypeVar('T')  # tipo de entrada
R = TypeVar('R')  # tipo de salida

class Tick(ABC, Generic[T, R]):
    @abstractmethod
    def on_tick(self, val: T) -> R:
        """Procesa un nuevo tick y, opcionalmente, devuelve un resultado."""
        pass

### Ventana Deslizante: la estructura de datos fundamental

In [9]:
from collections import deque
from typing import Deque, Optional
import numpy as np

class DequeWindow(Tick[T, Optional[T]], Generic[T]):
    def __init__(self, n: int):
        self._data: Deque[T] = deque(maxlen=n)

    def on_tick(self, val: T) -> Optional[T]:
        """Agrega un valor y devuelve el más antiguo que salió de la ventana (si lo hay)."""
        dropped = None
        if self.is_full():
            dropped = self._data[0]
        self._data.append(val)
        return dropped

    def is_full(self) -> bool:
        return self._data.maxlen == len(self._data)

    def append_left(self, val: T) -> Optional[T]:
        dropped = None
        if self.is_full():
            dropped = self._data[-1]
        self._data.appendleft(val)
        return dropped

    def to_numpy(self) -> np.ndarray:
        return np.array(self._data)

    def __repr__(self) -> str:
        cls_name = self.__class__.__name__
        return f"{cls_name}(capacity={self._data.maxlen}, values={list(self._data)})"

In [10]:
w = DequeWindow(3)
w

DequeWindow(capacity=3, values=[])

In [11]:
w = DequeWindow(3)
w.on_tick(1)
w.on_tick(2)
w.on_tick(3)
w

DequeWindow(capacity=3, values=[1, 2, 3])

In [12]:
w.on_tick(4)
w

DequeWindow(capacity=3, values=[2, 3, 4])

In [13]:
for i in range(1000000):
    w.on_tick(i)
w

DequeWindow(capacity=3, values=[999997, 999998, 999999])

### Ventana Basada en Arreglos

In [14]:
class NumpyWindow(Tick[T, Optional[T]]):
    def __init__(self, n: int, dtype=np.float64):
        if n <= 0:
            raise ValueError("Capacity must be positive.")
        self._capacity = n
        self._data = np.zeros(n, dtype=dtype)
        self._size = 0

    def on_tick(self, val: float) -> Optional[float]:
        dropped = None

        if self._size < self._capacity:
            self._data[self._size] = val
            self._size += 1
        else:
            dropped = self._data[0]
            # desplazar a la izquierda en el mismo arreglo
            for i in range(1, self._capacity):
                self._data[i - 1] = self._data[i]
            self._data[-1] = val

        return dropped

    def __getitem__(self, idx: int) -> float:
        """Acceso por índice (0 = el más antiguo)."""
        if not 0 <= idx < self._size:
            raise IndexError("Index out of range.")
        return self._data[idx]

    def is_full(self) -> bool:
        return self._size == self._capacity

    def values(self) -> np.ndarray:
        return self._data[:self._size]

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(capacity={self._capacity}, size={self._size}, values={self.values().tolist()})"

### Comparación de Velocidad: Ventana Numpy vs. Ventana Deque

In [15]:
def benchmark_window(window, n):
    for i in range(n):
        window.on_tick(i)

window_size = 10
n = 1000000

In [16]:
%%time
benchmark_window(NumpyWindow(window_size), n)

CPU times: user 758 ms, sys: 3.49 ms, total: 762 ms
Wall time: 762 ms


In [17]:
%%time
benchmark_window(DequeWindow(window_size), n)

CPU times: user 132 ms, sys: 681 µs, total: 133 ms
Wall time: 133 ms


## Retornos Logarítmicos en Streaming

### Repaso

In [18]:
ts = [100, 120, 100]
log_returns = [
    np.log(ts[1] / ts[0]),
    np.log(ts[2] / ts[1])
]
log_returns

[np.float64(0.1823215567939546), np.float64(-0.1823215567939546)]

In [19]:
class LogReturn(Tick[float, Optional[float]]):
    def __init__(self):
        self._window = NumpyWindow(2)

    def on_tick(self, val: float) -> Optional[float]:
        self._window.on_tick(val)
        if self._window.is_full():
            return np.log(self._window[1] / self._window[0])
        else:
            return None

    def __repr__(self) -> str:
        cls_name = self.__class__.__name__
        return f"{cls_name}(window={self._window})"

In [20]:
f = LogReturn()
f.on_tick(100.0)
f

LogReturn(window=NumpyWindow(capacity=2, size=1, values=[100.0]))

In [21]:
v = f.on_tick(120.0)
v

np.float64(0.1823215567939546)

In [22]:
v = f.on_tick(100.0)
v

np.float64(-0.1823215567939546)

In [23]:
f

LogReturn(window=NumpyWindow(capacity=2, size=2, values=[120.0, 100.0]))

### Rezagos Autorregresivos del Retorno Logarítmico en Streaming

### Repaso

In [24]:
time_series = [0.1, -0.2, -0.3]
lag_1 = time_series[-1]
lag_1

-0.3

In [25]:
lag_3 = time_series[-3]
lag_3

0.1

In [26]:
class LogReturnLags(Tick[float, torch.Tensor]):
    def __init__(self, no_lags: int):
        self._lags = DequeWindow(no_lags)
        self._log_return = LogReturn()

    def on_tick(self, val: float) -> Optional[torch.Tensor]:
        log_ret = self._log_return.on_tick(val)
        if log_ret is not None:
            self._lags.append_left(log_ret)
            return torch.tensor(self._lags.to_numpy(), dtype=torch.float32) if self._lags.is_full() else None
        else:
            return None

    def __repr__(self) -> str:
        cls_name = self.__class__.__name__
        return f"{cls_name}(lags={self._lags}, log_return={self._log_return})"

In [27]:
lags = LogReturnLags(3)
v = lags.on_tick(90)
lags

LogReturnLags(lags=DequeWindow(capacity=3, values=[]), log_return=LogReturn(window=NumpyWindow(capacity=2, size=1, values=[90.0])))

In [28]:
lags.on_tick(100)
lags

LogReturnLags(lags=DequeWindow(capacity=3, values=[np.float64(0.10536051565782635)]), log_return=LogReturn(window=NumpyWindow(capacity=2, size=2, values=[90.0, 100.0])))

In [29]:
lags.on_tick(150)
lags

LogReturnLags(lags=DequeWindow(capacity=3, values=[np.float64(0.4054651081081644), np.float64(0.10536051565782635)]), log_return=LogReturn(window=NumpyWindow(capacity=2, size=2, values=[100.0, 150.0])))

In [30]:
lags.on_tick(110)

tensor([-0.3102,  0.4055,  0.1054])

In [31]:
[np.log(110/150), np.log(150/100), np.log(100/90)]

[np.float64(-0.3101549283038396),
 np.float64(0.4054651081081644),
 np.float64(0.10536051565782635)]

### Pasar las Variables en Streaming al Modelo

In [32]:
# el modelo usa los rezagos 1 y 3: las posiciones 0 y 2 del vector
X = lags.on_tick(160)[[0, 2]]
with torch.no_grad():
    y_hat = model(X)
y_hat

tensor([0.0038])

## Construir el Sistema de Trading

### Usar Decimal para Representar Dinero

In [33]:
val = 0.1
total = 0.0
for i in range(10):
    total += val
total

0.9999999999999999

In [34]:
from decimal import Decimal
dp = Decimal('0.2')
val = Decimal(0.1).quantize(dp)
total = Decimal(0.0).quantize(dp)
for i in range(10):
    total += val
total

Decimal('1.0')

In [35]:
from dataclasses import dataclass

@dataclass(frozen=True)
class Order:
    sym: str
    signed_qty: Decimal

    def __str__(self) -> str:
        sign = "LONG" if self.signed_qty > 0 else "SHORT"
        return f"Order({sign} {self.signed_qty} {self.sym})"

In [36]:
def is_long(x: Decimal) -> bool:
    return x > Decimal(0)

@dataclass(frozen=True)
class Trade:
    sym: str
    signed_qty: Decimal
    price: Decimal
    pnl: Decimal

    def is_long(self) -> bool:
        return is_long(self.signed_qty)

In [37]:
@dataclass
class Position:
    sym: str
    signed_qty: Decimal
    price: Decimal

    def close(self) -> "Order":
        return Order(self.sym, -self.signed_qty)

    def is_long(self) -> bool:
        return is_long(self.signed_qty)

    def unrealized_pnl(self, current_price: Decimal) -> Decimal:
        return (current_price - self.price) * self.signed_qty

In [38]:
class Account(ABC):
    @abstractmethod
    def balance(self) -> Decimal:
        pass

    @abstractmethod
    def get_position(self, sym: str) -> Optional[Position]:
        pass

In [39]:
from typing import Dict, List

class TestAccount(Account):
    """Cuenta simulada para pruebas u operación en papel (paper trading)."""

    def __init__(self, _balance: Decimal) -> None:
        self._balance = _balance
        self._positions: Dict[str, Position] = {}
        self._trades: List[Trade] = []

    def balance(self) -> Decimal:
        return self._balance

    def get_position(self, sym) -> Optional[Position]:
        return self._positions.get(sym)

    def __repr__(self) -> str:
        return f"TestAccount(balance={self._balance}, positions={self._positions}, trades={self._trades})"

In [40]:
acc = TestAccount(Decimal(50.0))
acc

TestAccount(balance=50, positions={}, trades=[])

### Modelar un Exchange

In [41]:
class Exchange(Account):
    """Clase base abstracta que representa un exchange o bróker."""

    @abstractmethod
    def market_order(self, sym: str, signed_qty: Decimal, price: Decimal) -> Trade:
        """Ejecuta una orden a mercado y devuelve el Trade resultante."""
        pass

In [42]:
class TestExchange(Exchange):
    _account: TestAccount

    def __init__(self, account: TestAccount):
        self._account = account

    def market_order(self, sym: str, signed_qty: Decimal, price: Decimal) -> "Trade":
        # Actualizar saldo y posición
        trade = self._update_position(sym, signed_qty, price)
        self._account._balance += trade.pnl
        self._account._trades.append(trade)
        return trade

    def _update_position(self, sym: str, signed_qty, price: Decimal) -> Trade:
        position = self._account._positions.pop(sym, None)
        pnl = Decimal(0.0)
        if position is not None:
            pnl = position.unrealized_pnl(price)       # La orden cierra la posición: la ganancia o pérdida se realiza
        else:
            self._account._positions[sym] = Position(sym, signed_qty, price)
        return Trade(sym, signed_qty, price, pnl)

    def balance(self) -> Decimal:
        return self._account.balance()

    def get_position(self, sym) -> Optional[Position]:
        return self._account.get_position(sym)

    def __repr__(self) -> str:
        return f"TestExchange(balance={self.balance()}, positions={self._account._positions}, trades={len(self._account._trades)})"

### Abrir una Posición

In [43]:
exchange = TestExchange(TestAccount(Decimal(50.0)))

price = Decimal(10)
qty = Decimal(5.0)
exchange.market_order('BTCUSDT', qty, Decimal(price))

Trade(sym='BTCUSDT', signed_qty=Decimal('5'), price=Decimal('10'), pnl=Decimal('0'))

In [44]:
exchange

TestExchange(balance=50, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('5'), price=Decimal('10'))}, trades=1)

### Cerrar la Posición

In [45]:
price = Decimal(15.0)
exchange.market_order('BTCUSDT', -qty, price)

Trade(sym='BTCUSDT', signed_qty=Decimal('-5'), price=Decimal('15'), pnl=Decimal('25'))

In [46]:
exchange

TestExchange(balance=75, positions={}, trades=2)

### Construir la API de la Estrategia

In [47]:
class Strategy(ABC):
    @abstractmethod
    def on_tick(self, price: float, account: Account) -> Optional[List[Order]]:
        pass

### Implementar Nuestra Estrategia

In [48]:
import torch.nn as nn

class BasicTakerStrat(Strategy):
    def __init__(self,
                 sym: str,
                 model: nn.Module,
                 log_return_lags: LogReturnLags,
                 lags: List[int],
                 scale_factor: Decimal = None) -> None:
        self.sym = sym
        self.model = model
        self.log_return_lags = log_return_lags
        self.lags = [lag - 1 for lag in lags]       # Posiciones de los rezagos que usa el modelo
        self.scale_factor = Decimal(scale_factor if scale_factor is not None else 1.0)

    def _signed_compound_trade_size(self, y_hat: float, account: Account, cur_price: Decimal, position: Optional[Position]) -> Decimal:
        dir_signal = np.sign(y_hat)
        unrealized_balance = account.balance() + (position.unrealized_pnl(cur_price) if position else Decimal(0.0))
        qty = unrealized_balance / cur_price
        return Decimal(dir_signal) * qty * self.scale_factor

    def _create_orders(self, y_hat: torch.Tensor, account: Account, price: Decimal) -> List[Order]:
        position = account.get_position(self.sym)
        signed_trade_size = self._signed_compound_trade_size(y_hat.item(), account, price, position)
        open_order = Order(self.sym, signed_trade_size)
        if position is None:
            return [open_order]
        if position.is_long() == is_long(signed_trade_size):
            return []                                # La dirección no cambia: la posición se mantiene
        return [position.close(), open_order]        # La dirección cambia: cerrar y abrir la contraria

    def on_tick(self, price: float, account: Account) -> List[Order]:
        X = self.log_return_lags.on_tick(price)
        if X is not None:
            with torch.no_grad():
                y_hat = self.model(X[self.lags])
                return self._create_orders(y_hat, account, Decimal(price))
        return []

In [49]:
# Ventana para transmitir los retornos logarítmicos rezagados
lags = LogReturnLags(3)
# Crear la cuenta
acc = TestAccount(Decimal(100.0))
# Crear la estrategia: el modelo usa los rezagos 1 y 3
strat = BasicTakerStrat('BTCUSDT', model, lags, [1, 3])

# Primer intervalo de 12 horas
strat.on_tick(10.0, acc)

[]

In [50]:
# Segundo intervalo de 12 horas
strat.on_tick(120.0, acc)

[]

In [51]:
# Tercer intervalo de 12 horas
strat.on_tick(90.0, acc)

[]

In [52]:
# Cuarto intervalo de 12 horas: ya hay 3 retornos y el modelo puede decidir
orders = strat.on_tick(100, acc)
orders

[Order(sym='BTCUSDT', signed_qty=Decimal('1'))]

### Ejecutar la Orden

In [53]:
exchange = TestExchange(acc)
order = orders[0]
exchange.market_order(order.sym, order.signed_qty, 100)

Trade(sym='BTCUSDT', signed_qty=Decimal('1'), price=100, pnl=Decimal('0'))

In [54]:
exchange

TestExchange(balance=100, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('1'), price=100)}, trades=1)

In [55]:
# la predicción cambia de signo: una orden cierra la posición y otra abre la contraria
orders = strat.on_tick(115, acc)
orders

[Order(sym='BTCUSDT', signed_qty=Decimal('-1')),
 Order(sym='BTCUSDT', signed_qty=Decimal('-1'))]

In [56]:
for order in orders:
    print(exchange.market_order(order.sym, order.signed_qty, 115))
exchange

Trade(sym='BTCUSDT', signed_qty=Decimal('-1'), price=115, pnl=Decimal('15'))
Trade(sym='BTCUSDT', signed_qty=Decimal('-1'), price=115, pnl=Decimal('0'))


TestExchange(balance=115, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('-1'), price=115)}, trades=3)

In [57]:
orders = strat.on_tick(100, acc)
orders

[Order(sym='BTCUSDT', signed_qty=Decimal('1')),
 Order(sym='BTCUSDT', signed_qty=Decimal('1.3'))]

In [58]:
for order in orders:
    print(exchange.market_order(order.sym, order.signed_qty, 100))
exchange

Trade(sym='BTCUSDT', signed_qty=Decimal('1'), price=100, pnl=Decimal('15'))
Trade(sym='BTCUSDT', signed_qty=Decimal('1.3'), price=100, pnl=Decimal('0'))


TestExchange(balance=130, positions={'BTCUSDT': Position(sym='BTCUSDT', signed_qty=Decimal('1.3'), price=100)}, trades=5)

In [59]:
# la predicción no cambia de signo: no hay órdenes y la posición se mantiene
strat.on_tick(101, acc)

[]

---
# E · Ejecución
## Mismo Modelo, Misma Señal

In [60]:
# antes de conectar nada: el sistema en streaming debe decidir exactamente lo mismo que el backtest del notebook 2
import polars as pl
from datetime import timedelta

sym, time_interval, target = 'BTCUSDT', '12h', 'close_log_return'
features = [f'{target}_lag_1', f'{target}_lag_3']

ohlc = pl.read_csv(f"{sym}_{time_interval}_ohlc.csv", try_parse_dates=True).sort('datetime')
ts = research.add_log_return_features(ohlc, 'close', 1, max_no_lags=4).drop_nulls()
_, _, trades = research.timeseries_split(ts)
trades = research.add_trade_log_returns(research.add_model_predictions(trades, model, features))
trades.select('datetime', 'close', 'y_hat', 'dir_signal')

datetime,close,y_hat,dir_signal
datetime[μs],f64,f64,f64
2025-11-13 12:00:00,99654.3,0.000478,1.0
2025-11-14 00:00:00,96134.6,0.000004,1.0
2025-11-14 12:00:00,94544.9,0.000722,1.0
2025-11-15 00:00:00,95657.3,-0.000094,-1.0
2025-11-15 12:00:00,95560.3,-0.000352,-1.0
…,…,…,…
2026-10-29 12:00:00,77962.6,0.00046,1.0
2026-10-30 00:00:00,77345.9,0.000472,1.0
2026-10-30 12:00:00,77312.6,0.000111,1.0


In [61]:
# las mismas velas, una a una, por los componentes de este notebook
lags = LogReturnLags(3)
stream = []
for bar in ohlc.iter_rows(named=True):
    X = lags.on_tick(bar['close'])                  # la vela acaba de cerrar
    if X is not None:
        with torch.no_grad():
            y_hat = model(X[[0, 2]])
        # la decisión vale para la vela siguiente
        stream.append({'datetime': bar['datetime'] + timedelta(hours=12), 'stream_signal': float(np.sign(y_hat.item()))})

check = trades.join(pl.DataFrame(stream), on='datetime')
# velas comparadas y fracción en la que las dos señales coinciden
len(check), (check['dir_signal'] == check['stream_signal']).mean()

(705, 1.0)

## Del Exchange Simulado a Binance Demo

In [62]:
# el bot son estas mismas piezas, en archivos .py dentro de la carpeta bot/:
# bot.py            -> la estrategia: cada 12 horas, C (velas) -> A (modelo) -> regla del notebook 2 -> E (orden)
# market_data.py    -> las velas ya cerradas del mercado real (datos públicos, sin claves)
# trading_system.py -> Order, Trade, el exchange simulado y la regla que lleva la posición al objetivo
# binance_demo.py   -> el exchange de verdad: la cuenta demo de futuros de Binance
# comun.py          -> el bucle: esperar el cierre de la vela -> decidir -> registrar -> avisar
# avisos.py         -> los mensajes por Telegram
# config.json       -> capital, apalancamiento y banda

### Paso 1: una decisión con datos reales y exchange simulado (no necesita claves)

In [63]:
%run bot/bot.py --dry-run --once

2026-11-26 16:14:23  MODO SIMULADO: datos reales, sin órdenes. Capital inicial: 1,000.00 USDT | apalancamiento 1x


2026-11-26 16:14:23  Avisos por Telegram: sin configurar (python bot/bot.py --telegram)


2026-11-26 16:14:24  2026-11-26 00:00 | cierre 60,686.8 | predicción -0.006% (CORTO) | objetivo -100.0% | posición -100.1% | capital 999.40 | orden -0.0165 a 60,686.8


In [64]:
# cada decisión queda registrada: precio, predicción, posición objetivo, posición final, capital y orden
pl.read_csv('bot/registros/simulado/decisiones.csv')

datetime,close,y_hat,target,position,equity,order_qty,fill_price,cost
str,f64,f64,f64,f64,f64,f64,f64,f64
"""2026-11-26 00:00:00""",60686.8,-0.000058,-1.0,-1.0006,999.4,-0.016478,60686.8,0.6


### Paso 2: guardar las claves

In [65]:
# 1. copia bot/.env.example y nómbralo bot/.env
# 2. pega ahí la API Key y la Secret Key que creaste DENTRO de Binance Demo Trading (futuros)
# 3. nunca pegues las claves en un notebook, en un chat ni en una captura de pantalla: el archivo .env no se sube a GitHub

### Paso 3: comprobar la conexión (no envía órdenes)

In [ ]:
%run bot/bot.py --check

### Paso 4: enviar una orden de prueba (compra y vende el mínimo)

In [ ]:
%run bot/bot.py --test-order

### Paso 5: activar los avisos por Telegram

In [68]:
# 1. en Telegram, busca @BotFather, envía /newbot y pega el token en bot/.env como TELEGRAM_BOT_TOKEN
# 2. escríbele cualquier mensaje a tu bot y ejecuta la celda: te muestra tu chat_id
# 3. pega el chat_id en bot/.env como TELEGRAM_CHAT_ID y ejecuta la celda otra vez: te llega un mensaje de prueba

In [ ]:
%run bot/bot.py --telegram

### Paso 6: la primera decisión en la cuenta demo

In [ ]:
%run bot/bot.py --once

### Paso 7: encender el bot

In [71]:
# en una terminal abierta en la carpeta metodo_CASTLE_castlequant.com (la terminal debe quedar abierta y el computador sin suspenderse):
#
#     python bot/bot.py
#
# decide con la última vela cerrada y luego espera cada cierre: 00:00 y 12:00 UTC (7 p. m. y 7 a. m. en Colombia)
# para detenerlo: Ctrl + C. La posición abierta NO se cierra sola; para cerrarla:
#
#     python bot/bot.py --close-position

In [72]:
# fin del notebook 3: el método completo
# C · Captura -> A · Alfa -> S · Simulación -> T · Tamaño -> L · Libre de sesgos -> E · Ejecución
# el bot es funcional. Para que además sea rentable hay que cambiar la pieza A y volver a pasar S, T y L